# 00 — Check the GLOF setup
**Salon Bikram Thapa · 17624860 · Machine Learning - Coventry University**

This notebook checks package versions and source-file access using the **Python (GLOF)** environment. No model fitting is performed.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/model_analysis.py').is_file())
DATA = ROOT.parent / 'outputs' / 'GLOF_MSc_datasets'
REPORTS = ROOT / 'reports'
PROCESSED = ROOT / 'data' / 'processed'
REPORTS.mkdir(exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'verification' / 'matplotlib'))
print('Project:', ROOT)


## Check installed packages
These cover modelling, maps, climate files, plotting and notebooks. We record the actual versions for reproducibility.

In [ ]:
import importlib, platform, shutil
packages = ['numpy','pandas','scipy','sklearn','geopandas','shapely','pyproj','rasterio','xarray','netCDF4','matplotlib','seaborn','nbformat','nbclient','ipykernel']
versions = {name: getattr(importlib.import_module(name), '__version__', 'imported') for name in packages}
from jupyter_client.kernelspec import KernelSpecManager
kernel = KernelSpecManager().get_kernel_spec('glof')
assert Path(kernel.argv[0]).resolve() == Path(sys.executable).resolve(), 'Restart with Python (GLOF).'
report = {'python':sys.executable,'platform':platform.platform(),'versions':versions,'kernel':kernel.display_name,'free_disk_GiB':round(shutil.disk_usage(ROOT).free/2**30,1)}
print(json.dumps(report, indent=2))
(REPORTS / 'setup_check.json').write_text(json.dumps(report, indent=2))

## Check access to the core inventories and optional climate data
We read the lake files and a small climate sample, rather than loading the global climate grid into memory.

In [ ]:
import geopandas as gpd
import xarray as xr
for folder in ['00_ICIMOD_base_2015','01_ICIMOD_PDGL']:
    files = list((DATA / folder).rglob('*.shp'))
    assert len(files) == 1, files
    frame = gpd.read_file(files[0], rows=5)
    assert len(frame) == 5 and frame.crs is not None
    print(folder, 'readable; CRS:', frame.crs)
nc_files = list((DATA / '05_ERA5_Land').rglob('*.nc'))
assert len(nc_files) == 1
with xr.open_dataset(nc_files[0]) as ds:
    print('Climate dimensions:', dict(ds.sizes))
    for v in ['t2m','tp']:
        assert v in ds
        print(v, 'sample:', ds[v].isel({dim:0 for dim in ds[v].dims}).load().values)
print('Setup checks passed. Continue to notebook 01.')